# Noise-Aware Variational Quantum Policy on CartPole

**Farah Mohamed Abdou Ahmed** | Interview demo for MSc in Quantum Computing & QML, University of Guelph

**Research question:** At matched parameter budget, can a variational quantum policy match a classical MLP policy on CartPole — and how does it degrade under realistic gate noise?

**Why this question:** I completed a Quantum RL internship at Nile University (Aug-Dec 2024). What bothered me afterward was that almost no QRL paper reports performance under realistic noise *with matched-budget classical baselines*. This notebook revisits the problem with that methodological discipline.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml

from src import quantum_policy as qp
from src import classical_policy as cp
from src import evaluate as ev
from src.noise import NOISE_LEVELS, NOISE_LABELS

RESULTS = '../results'
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## 2. The quantum policy — what it looks like

A 4-qubit variational circuit with angle encoding and 2 layers of RY+RZ rotations plus a CNOT ring.

In [ ]:
circuit = qp.make_policy(noise_prob=0.0)
params = qp.init_params(seed=0)
sample_state = np.array([0.1, -0.2, 0.05, 0.3])

# Draw the circuit
fig, ax = qml.draw_mpl(circuit)(sample_state, params)
plt.show()

In [ ]:
# Parameter count check — quantum and classical match exactly
print(f'Quantum policy parameters:   {qp.count_parameters()}')
print(f'Classical MLP parameters:    {cp.count_parameters()}')

## 3. Figure 1 — Noiseless learning curves

5 seeds each. Mean ± std across seeds.

In [ ]:
# Load results saved by experiments/run_all.py
classical = pd.read_csv(f'{RESULTS}/classical_noiseless.csv', index_col='episode').values.T
quantum   = pd.read_csv(f'{RESULTS}/quantum_noiseless.csv',  index_col='episode').values.T

def smooth_curve(arr, w=20):
    out = np.zeros_like(arr, dtype=float)
    for i in range(arr.shape[0]):
        out[i] = np.convolve(arr[i], np.ones(w)/w, mode='same')
    return out

c_s = smooth_curve(classical)
q_s = smooth_curve(quantum)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(c_s.shape[1])

ax.plot(x, c_s.mean(0), label='Classical MLP (16 params)', color='C0', linewidth=2)
ax.fill_between(x, c_s.mean(0)-c_s.std(0), c_s.mean(0)+c_s.std(0), color='C0', alpha=0.2)

ax.plot(x, q_s.mean(0), label='Quantum VQP (16 params)', color='C1', linewidth=2)
ax.fill_between(x, q_s.mean(0)-q_s.std(0), q_s.mean(0)+q_s.std(0), color='C1', alpha=0.2)

ax.set_xlabel('Episode')
ax.set_ylabel('Episode reward (smoothed)')
ax.set_title('Noiseless CartPole — matched parameter budget')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS}/fig1_noiseless_curves.png', dpi=150)
plt.show()

In [ ]:
# Paired t-test on final-100-episode performance
q_finals = quantum[:, -100:].mean(axis=1)
c_finals = classical[:, -100:].mean(axis=1)
t, p = ev.paired_t_test(q_finals, c_finals)

print(f'Final-100 reward: classical {c_finals.mean():.1f} ± {c_finals.std():.1f}')
print(f'Final-100 reward: quantum   {q_finals.mean():.1f} ± {q_finals.std():.1f}')
print(f'Paired t-test:   t = {t:.3f},  p = {p:.4f}')

## 4. Figure 2 — Noise sweep (the centerpiece)

How does the quantum policy degrade as gate noise increases?

In [ ]:
noise_results = {}
for p_val in NOISE_LEVELS:
    fname = f'{RESULTS}/quantum_noiseless.csv' if p_val == 0.0 else f'{RESULTS}/quantum_noise_p{p_val}.csv'
    if os.path.exists(fname):
        arr = pd.read_csv(fname, index_col='episode').values.T
        noise_results[p_val] = arr[:, -100:].mean(axis=1)   # final-100 per seed

classical_final = c_finals.mean()

fig, ax = plt.subplots(figsize=(8, 5))
x = list(noise_results.keys())
y_mean = [noise_results[p].mean() for p in x]
y_std  = [noise_results[p].std()  for p in x]

ax.errorbar(x, y_mean, yerr=y_std, fmt='o-', color='C1', linewidth=2, capsize=4,
            label='Quantum VQP')
ax.axhline(classical_final, linestyle='--', color='C0', label=f'Classical baseline ({classical_final:.0f})')
ax.set_xlabel('Gate noise probability p')
ax.set_ylabel('Final-100-episode reward')
ax.set_title('Noise sweep: how the quantum policy degrades')
ax.set_xscale('symlog', linthresh=0.001)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS}/fig2_noise_sweep.png', dpi=150)
plt.show()

## 5. Compute accounting

The hidden cost of quantum that most QRL papers omit.

In [ ]:
specs = qml.specs(circuit)(sample_state, params)
print('Quantum circuit specs:')
for k in ['num_used_wires', 'num_operations', 'num_trainable_params']:
    print(f'  {k}: {specs.get(k, "?")}')
if 'resources' in specs:
    print(f'  depth: {specs["resources"].depth}')

print('\nCost per gradient step:')
print(f'  Classical: 1 forward + 1 backward pass')
print(f'  Quantum:   1 + 2*16 = 33 circuit evaluations (parameter-shift rule)')

## 6. Honest conclusion

*[Fill these in with YOUR actual numbers after you run the experiments.]*

- **At matched 16-parameter budget on noiseless CartPole**, the quantum policy reached `XX%` of the classical policy's final reward (paired t-test p = `Y.YY` across 5 seeds).
- **Sample efficiency:** classical converged in roughly `A` episodes; quantum required roughly `B` episodes.
- **Noise tolerance:** the quantum policy maintained classical-comparable performance up to roughly `p = 0.00X`, and collapsed by `p = 0.0Y`. Real superconducting hardware in 2026 operates near `p = 0.001–0.01` per two-qubit gate, so this places today's hardware near the cliff.
- **Compute cost:** the quantum policy required `~33×` more forward passes per gradient step than classical, with wall-clock roughly `~60×` slower per episode in simulation.

**I do not claim quantum advantage on this task at this scale.** The contribution is the methodological framework: matched parameter budget, paired statistical tests, honest compute accounting, and a noise-degradation map. This is the empirical discipline I would bring to graduate work in the Quantum Computing Lab.

## 7. What I'd extend in the MSc

1. **Encoding sweep** (Subproject A): repeat across angle / amplitude / IQP / re-uploading encodings.
2. **Other environments**: FrozenLake (discrete), LunarLander (8-dim state), and a problem with quantum-native structure (e.g. quantum control).
3. **Error mitigation**: zero-noise extrapolation, probabilistic error cancellation — measure whether they buy back the noise cliff.
4. **Real hardware**: port to Qiskit Runtime, compare simulator predictions to actual IBM Quantum / IonQ runs.
5. **AI + Health crossover**: apply the same methodology to a quantum-kernel classifier on a small clinical tabular dataset (e.g. Parkinson's tremor features).